# 05 — Acquire weekly oil prices

Download the European Commission Weekly Oil Bulletin price-history workbook (2005 onward), then inventory its sheets before extraction.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from portugal_refining_resilience.config import get_paths
from portugal_refining_resilience.io import persist_dataframe, write_json

PATHS = get_paths(ROOT)
pd.set_option("display.max_columns", 100)

from portugal_refining_resilience.sources import download_file, load_source_manifest
from portugal_refining_resilience.validation import assert_unique
from portugal_refining_resilience.excel import workbook_inventory
from portugal_refining_resilience.prices import extract_weekly_prices


In [ ]:
sources = load_source_manifest(ROOT / "config" / "sources.yml")
url = sources["ec_weekly_oil_bulletin"]["history_xlsx"]
target = PATHS.raw / "ec_weekly_oil_bulletin" / "weekly_oil_bulletin_price_history.xlsx"
download_file(url, target)
inventory = workbook_inventory(target)
display(inventory[["sheet", "rows", "columns"]])
persist_dataframe(inventory, PATHS.provenance / "weekly_oil_bulletin_workbook_inventory.csv", key_columns=["sheet"])


In [ ]:
# Extraction is separated from download so a workbook layout change fails here,
# not silently inside the price models. extract_weekly_prices rejects unexpected
# units and missing country/product columns rather than guessing.
prices = extract_weekly_prices(target)
assert_unique(prices, ["date", "country", "product"])
persist_dataframe(
    prices,
    PATHS.interim / "weekly_oil_prices_tidy.csv",
    key_columns=["date", "country", "product"],
    metadata={
        "unit": "EUR per 1000 litres",
        "primary_measure": "price_without_tax_eur_per_1000l",
        "source": "EC Weekly Oil Bulletin price history",
    },
)
display(prices.groupby(["country", "product"]).agg(n_weeks=("date", "size"), first=("date", "min"), last=("date", "max")))

The historical workbook layout is controlled by the Commission and may change. Extraction is therefore deliberately separated from download. Before changing parsers, preserve this inventory and inspect the relevant sheet previews.
